## Problem: LLM-Based Memory Deduplication

The memory extractor may generate the **same or similar memory multiple times** when the same information is provided repeatedly.

For example:

```text
Existing:
"User prefers Python for backend development."

New:
"User likes Python for backend development."
```

Although the wording is different, both represent the **same user preference**.


Therefore, an **LLM-based deduplication step** is needed to compare the new memory with existing memories and decide whether to: 
`create` / `update` / `skip`

---



## Approach: LLM-based deduplication

LLM-based deduplication means using an LLM to decide whether a newly extracted memory is already known or should modify an existing memory.

* **CREATE** → store as a new memory
* **UPDATE** → modify an existing memory
* **SKIP** → ignore the duplicate memory

> In this Approach LLM compares the extracted preferances with existing one, let LLM to decide whether it should be new or existing one. based on the LLM's response for each new preference we either CREATE or UPDATE or SKIP the new prefernces.

### Example

**Existing memory:**

> User prefers Python for backend development.

**New memory:**

> User likes using Python to build backend applications.

**LLM decision:** `SKIP` — same meaning, so don't create a duplicate.

Another example:

**Existing memory:**

> User prefers Python.

**New memory:**

> User now prefers TypeScript for backend development.

**LLM decision:** `UPDATE` — the new information changes the existing preference.

---


## Code from Previous Flow
#### File: [21_LTM_long_term_memory_implementation](21_LTM_long_term_memory_implementation.ipynb)

The overall flow remains unchanged. The upgrade is only inside the `memory_extractor` node.

#### **Previous Flow:**

```text
    START
      ↓
    Memory Extractor
    ├── Extract
    └── Create
      ↓
    Memory Retriever
      ↓
    Memory-Aware Chat
      ↓
    END
```

#### **New Flow:**
```text
    START
    ↓
    Memory Extractor
    ├── Extract
    ├── Compare
    └── Create / Update / Skip
    ↓
    Memory Retriever
    ↓
    Memory-Aware Chat
    ↓
    END
```

In [16]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import uuid

# Long-Term Memory (LTM) imports
# BaseStore: Abstract interface for storing/retrieving memories across conversation sessions
# InMemoryStore: Default implementation that persists memories within a single process lifetime
from langgraph.store.base import BaseStore, SearchItem
from langgraph.store.memory import InMemoryStore
from langchain_core.runnables import RunnableConfig

In [15]:
# State definition for LTM-aware chatbot
# - messages: conversation history for context
# - memories: retrieved long-term memories that persist across different conversation sessions/threads
class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    memories: list[str]

In [17]:
# Structured output model for LTM extraction
# Using Pydantic ensures LLM returns parseable, consistent memory items
# This enables reliable storage and retrieval from InMemoryStore
class ExtractPreferencesModel(BaseModel):
    is_exist: list[bool] = Field(description="for each preferences need a Boolean value, the value would be True if preference/details already exist otherwize False") 
    preferences: list[str] = Field(default=[], description="preferences/details which can consider for long term memory")

In [18]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-3.1-flash-lite",
    temperature=0.6
)

In [19]:
# MEMORY_EXTRACTOR_PROMPT

MEMORY_EXTRACTOR_PROMPT = """You are a long-term memory extraction agent.

Analyze the user's message and identify meaningful, user-specific information that is useful for future conversations.

Existing preferences/memories:
{existing_preferences}

For each preference or detail extracted from the user's message:
- Add it to `preferences` as a short, clear statement.
- Add a corresponding boolean to `is_exist`.
- Set `is_exist` to True if the preference/detail already exists or is semantically similar to an existing memory.
- Set `is_exist` to False if it is new information.
- If new information updates or changes an existing preference, consider it existing and set `is_exist` to True.

Store only meaningful information such as:
- Preferences
- Skills
- Interests
- Goals
- Ongoing projects
- Important facts

Do not store temporary, irrelevant, or conversational information.

Ensure `is_exist` and `preferences` have the same length and corresponding indexes."""

In [21]:
# MEMORY_AWARE_CHATBOT_PROMPT

MEMORY_AWARE_CHATBOT_PROMPT = """You are a helpful AI assistant.

Answer the user's message naturally.

Use the retrieved long-term memories when they are relevant to the user's request.
Do not mention the memory system or tell the user that you retrieved memories.

Existing Preferences:
{preferences}

Use these preferences to personalize your response when relevant.

If no relevant memories are available, answer using the current conversation context and your general knowledge.

Do not invent information that is not available."""

In [22]:
def memory_extractor(state: State, config: RunnableConfig, *, store: BaseStore):
    # Hierarchical namespace for LTM isolation
    # ("users", user_id, "details") - enables multi-user LTM segregation
    # Each user has isolated memory namespace, preventing cross-user memory leakage
    namespace = ("users", config["configurable"]["user_id"], "details")

    current_message = state["messages"][-1]

    existing_preferences = [pref.value["data"] for pref in store.search(namespace)]

    prompt = MEMORY_EXTRACTOR_PROMPT.format(
        existing_preferences=existing_preferences
    )

    # Use structured output to extract LTM-relevant information consistently
    structured_llm = llm.with_structured_output(ExtractPreferencesModel)

    resp = structured_llm.invoke([SystemMessage(content=prompt), current_message])

    # Store each extracted preference as a separate memory entry
    for it in zip(resp.preferences, resp.is_exist):
        if not it[1]:
            memory_id = str(uuid.uuid4())
            store.put(namespace=namespace, key=memory_id, value={"data": it[0]})
        
    return {}

In [23]:
def memory_retriver(state: State, config: RunnableConfig, *, store: BaseStore):
    # Retrieve all memories from user's namespace
    # store.search() retrieves all stored memories for the current user
    # This enables the chatbot to access previously learned preferences across sessions
    namespace = ("users", config["configurable"]["user_id"], "details")

    memories = store.search(namespace)

    memories_list = []

    # Extract memory data from store items
    for memory in memories:
        memories_list.append(memory.value["data"])

    # Return extracted memories to state for use in chat node
    return {
        "memories" : memories_list
    }

In [24]:
def chat_node(state: State, config: RunnableConfig, store: BaseStore):
    messages = state["messages"]
    
    # Retrieved LTM memories are injected into the system prompt
    # This makes the LLM aware of user preferences/details from previous sessions
    # The LLM uses these memories to personalize responses without explicit memory references
    memories = state["memories"]

    prompt = MEMORY_AWARE_CHATBOT_PROMPT.format(preferences=memories)

    resp = llm.invoke([SystemMessage(content=prompt)] + messages)

    return {
        "messages": [resp]
    }

In [25]:
# LTM workflow: Extract → Retrieve → Personalized Response
# 1. memory_extractor: Parses current message for LTM-worthy information and stores it
# 2. memory_retriver: Fetches all user memories from previous conversations
# 3. chat_node: Uses memories to generate context-aware, personalized responses
builder = StateGraph(State)\
    .add_node("memory_extractor", memory_extractor)\
    .add_node("memory_retriver", memory_retriver)\
    .add_node("chat_node", chat_node)\
    .add_edge(START, "memory_extractor")\
    .add_edge("memory_extractor", "memory_retriver")\
    .add_edge("memory_retriver", "chat_node")\
    .add_edge("chat_node", END)

In [26]:
# Pass InMemoryStore to graph compilation
# The store is injected into node functions via RunnableConfig, enabling LTM access
# This allows memory_extractor to write and memory_retriever to read from persistent storage
store = InMemoryStore()

graph = builder.compile(store=store)

In [27]:
# graph

In [28]:
# Graph config with user_id for LTM isolation
# user_id is the key to multi-user LTM: each user gets separate memory namespace
# Different user_ids prevent memory cross-contamination in multi-user systems
config = {
    "configurable": {
        "user_id": "user-1"
    }
}

In [29]:
# Test 1: First interaction - extracting and storing LTM
# This message provides context about the user that should be extracted and stored
msg = "my name is bhavin"

resp = graph.invoke({"messages": [HumanMessage(content=msg)]}, config)

print(resp["messages"][-1])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


content=[{'type': 'text', 'text': "It's nice to meet you, Bhavin! How can I help you today?", 'extras': {'signature': 'EnEKbwERTTIPpjZsN3uu26Igmz1ovTCjAODtvlWCKjy26GqbtzPo7Dh0odaYyUbbZ/RMh+JmczeIjBVFPi/YTEpkSz8kR258K6Zy+/b6XgcUa0ya3acPCLwZbHgMDIpFOrwNe9f2X11ECSOso1TUDM1CkQ=='}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a02b06-4b32-7c71-926a-5dfe5e317c30-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 111, 'output_tokens': 18, 'total_tokens': 129, 'input_token_details': {'cache_read': 0}}


In [38]:
# Test 2: Subsequent interaction - LTM enables personalized response
# Even though this question doesn't mention the user's name, LTM retrieves "my name is bhavin"
# The chatbot can personalize response using stored memory from previous conversation
msg = "i'm learning python"

resp = graph.invoke({"messages": [HumanMessage(content=msg)]}, config)

print(resp["messages"][-1])

content=[{'type': 'text', 'text': "That's great, Bhavin! Python is a fantastic language to learn, whether you're just starting out or looking to expand your programming skills.\n\nHow has your progress been so far? Are you working on any specific projects, or are you currently focusing on the fundamentals like data types and control flow? If you hit any roadblocks or need help explaining a concept, feel free to ask!", 'extras': {'signature': 'EnEKbwERTTIP2RQsqybPeC++TUx+6+GMzCWN42kIMY+Rn/dYBt3H9IU41m+9DdTJR/ikXKq7xm2VHWk2Tj7m51/n4ZpTfTd30jUUeizOlYGgE9B1LjnW7myKy/IpTQ6KwYRwBN+mkXEwEvZH8tOf7xPe5g=='}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a02b07-fc72-7cf0-915c-b2e0fddd04a0-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 119, 'output_tokens': 80, 'total_tokens': 199, 'input_token_details': {'cache_read': 0}}


### Issue in this workflow

hit below tab to generate the issue

In [40]:
# Test 3: again asking the same question 
# here it will again add the prefernce about the python and lambda function in store 
msg = "I like playing with data structure. can you tell me a little bit about array"

resp = graph.invoke({"messages": [HumanMessage(content=msg)]}, config)

print(resp["messages"][-1])

content=[{'type': 'text', 'text': 'That\'s great, Bhavin! Since you\'re learning Python and diving into data structures, arrays are a perfect place to start.\n\nIn computer science, an **array** is a data structure that stores a collection of elements, all of the same type, in contiguous memory locations. You can think of it like a row of lockers, where each locker has an index number, starting from 0. Because they are stored side-by-side in memory, you can access any specific element instantly if you know its index—this is what we call **constant time complexity**, or $O(1)$.\n\nNow, since you\'re using Python, there\'s a little nuance to keep in mind:\n\n1.  **Lists vs. Arrays:** In standard Python, what we usually call an "array" is actually a `list`. Python lists are very flexible because they can hold different data types and grow dynamically. However, they aren\'t technically "arrays" in the strict memory-layout sense used in languages like C or Java.\n2.  **The `array` module:**

### Verification

In [41]:
# Inspect stored LTM: Verify that memories were extracted and stored
# This demonstrates LTM persistence - all extracted memories remain in InMemoryStore
namespace = ("users", config["configurable"]["user_id"], "details")

store.search(namespace)

[Item(namespace=['users', 'user-1', 'details'], key='f2f455ab-1f6b-473e-a8c2-3b59887d613b', value={'data': "user's name is Bhavin"}, created_at='2026-08-22T19:50:27.881892+00:00', updated_at='2026-08-22T19:50:27.881897+00:00', score=None),
 Item(namespace=['users', 'user-1', 'details'], key='3e9fd16f-73cb-46b3-90f2-1d6bab0b304f', value={'data': 'Bhavin is learning Python'}, created_at='2026-08-22T19:51:20.665534+00:00', updated_at='2026-08-22T19:51:20.665540+00:00', score=None),
 Item(namespace=['users', 'user-1', 'details'], key='3a954bfb-e2ee-4808-a849-a3ed5074e041', value={'data': 'Bhavin is interested in data structures'}, created_at='2026-08-22T19:53:07.967701+00:00', updated_at='2026-08-22T19:53:07.967709+00:00', score=None)]

### Problem: Duplicate Long-Term Memories

When the **same user message is processed multiple times**, the memory extraction node creates and stores the same memory again with a new key.

For example:

```text
User: "I prefer Python"

Store:
1 → "User prefers Python"
2 → "User prefers Python"
3 → "User prefers Python"
```

This leads to **duplicate memories and unnecessary storage**.

The system needs a **deduplication or update mechanism** to detect whether a similar memory already exists before creating a new entry.

### Solutions
1. **Deterministic Key / Upsert** — Use a stable key so repeated memories overwrite the same entry.
2. **Exact Duplicate Check** — Compare the new memory with existing memories and skip exact matches.
3. **Semantic Similarity Check** — Use embeddings to detect memories with similar meaning.
4. **LLM-Based Deduplication** — Ask an LLM whether a new memory is duplicate, new, or an update.
5. **Hybrid Deduplication** — Combine exact matching, semantic search, and LLM reasoning for better accuracy.
